Вырезает из растра объект

In [1]:
import arcpy
from arcpy.sa import ExtractByMask
import os
import uuid

In [2]:
def erase_from_raster(
        in_raster: str | arcpy.Raster,
        erase_feature: str,
        out_path: str | None = None,
        snap_raster: str | arcpy.Raster | None = None
) -> str:
    """
    Вырезает область из растра по маске полигона (обратная обрезка).
    Поддерживает перезапись файла (in-place), используя временное хранилище.

    Args:
        in_raster: Путь к исходному растру или объект Raster.
        erase_feature: Векторный слой (полигон), который нужно "стереть" (превратить в NoData).
        out_path: Путь сохранения. Может совпадать с in_raster (перезапись).
        snap_raster: Растр для привязки ячеек (Snap Raster). Если None, берется in_raster.

    Returns:
        str: Путь к сохраненному файлу.
    """

    # 1. Получаем строковый путь к исходнику
    in_path_str = in_raster.catalogPath if hasattr(in_raster, "catalogPath") else str(in_raster)
    in_path_norm = os.path.normpath(in_path_str)

    # 2. Определяем, каким будет финальный путь (если out_path не задан, значит это исходник)
    final_dest_path = os.path.normpath(out_path) if out_path else in_path_norm

    # 3. Проверка: является ли операция перезаписью (совпадают ли пути)
    is_inplace = in_path_norm.lower() == final_dest_path.lower()

    # 4. Логика путей для временного файла
    if is_inplace:
        # Определяем директорию и имя
        out_dir = os.path.dirname(final_dest_path)
        base_name = os.path.basename(final_dest_path)
        name_no_ext, ext = os.path.splitext(base_name) # Получаем расширение автоматически

        # Создаем уникальное имя в ТОЙ ЖЕ папке
        # Используем родное расширение (ext), чтобы не потерять формат (.tif, .img и т.д.)
        temp_name = f"temp_{name_no_ext}_{uuid.uuid4().hex[:6]}{ext}"

        target_save_path = os.path.join(out_dir, temp_name)
        print(f"--- Режим перезаписи: временно сохраняем в {target_save_path} ---")
    else:
        target_save_path = final_dest_path

    # Настройки среды
    align_raster = snap_raster if snap_raster else in_raster

    try:
        with arcpy.EnvManager(snapRaster=align_raster, cellSize=align_raster, outputCoordinateSystem=align_raster, extent=align_raster):
            result = ExtractByMask(in_raster, erase_feature, extraction_area="OUTSIDE", analysis_extent=align_raster)
            result.save(target_save_path)
            arcpy.management.BuildPyramids(target_save_path, -1, "NONE", "NEAREST", "DEFAULT", 75, "OVERWRITE")

            # Снимаем блокировку
            del result

    except Exception as e:
        # Чистим мусор при ошибке
        if is_inplace and arcpy.Exists(target_save_path):
            try:
                arcpy.management.Delete(target_save_path)
            except:
                pass
        print(f"Ошибка в erase_from_raster: {e}")
        raise e

    # ФИНАЛИЗАЦИЯ (Swapping файлов)
    if is_inplace:
        try:
            print(f"--- Замена оригинального файла... ---")

            # Удаляем оригинал
            arcpy.management.Delete(in_path_norm)

            # Переименовываем временный файл в имя финала
            # Используем имя из final_dest_path, так как out_path может быть None
            arcpy.management.Rename(target_save_path, os.path.basename(final_dest_path))
            
        except Exception as e:
            print(f"КРИТИЧЕСКАЯ ОШИБКА при замене файла: {e}")
            print(f"Данные сохранены во временном файле: {target_save_path}")
            raise e

    print(f"Готово: {final_dest_path}")
    return final_dest_path

In [3]:
clip = r"I:\docs\maxim\prj\gis\learn\FloodBenineSmail\Maps\GDBs\TanDEMX.gdb\ocean_border"
rasters = [
    # r"I:\docs\maxim\prj\gis\learn\FloodBenineSmail\Maps\GDBs\TanDEMX.gdb\tdx_egm_fill",
    r"I:\docs\maxim\prj\gis\learn\FloodBenineSmail\Maps\GDBs\WorldCover.gdb\manning_n",
    r"I:\docs\maxim\prj\gis\learn\FloodBenineSmail\Maps\GDBs\FloodFactors.gdb\cost_horizont",
    # r"I:\docs\maxim\prj\gis\learn\FloodBenineSmail\Maps\GDBs\TanDEMX.gdb\slope_percent",
    # r"I:\docs\maxim\prj\gis\learn\FloodBenineSmail\Maps\Rasters\GHS_POP_E2030_GLOBE_R2023A_54009_100_V1_0.tif",
    # r"I:\docs\maxim\prj\gis\learn\FloodBenineSmail\Maps\GDBs\FloodFactorsNormal.gdb\Transformed_flood_river_RP100y_depth",
    # r"I:\docs\maxim\prj\gis\learn\FloodBenineSmail\Maps\GDBs\FloodFactorsNormal.gdb\Transformed_twi_x100",
    # r"I:\docs\maxim\prj\gis\learn\FloodBenineSmail\Maps\GDBs\FloodFactorsNormal.gdb\Transformed_slope_x100",
    # r"I:\docs\maxim\prj\gis\learn\FloodBenineSmail\Maps\GDBs\FloodFactorsNormal.gdb\Transformed_rainfall_intensity_mms",
    # r"I:\docs\maxim\prj\gis\learn\FloodBenineSmail\Maps\GDBs\FloodFactorsNormal.gdb\Transformed_SCS_CN",
    # r"I:\docs\maxim\prj\gis\learn\FloodBenineSmail\Maps\GDBs\FloodFactorsNormal.gdb\Transformed_pressure_acc",
    # r"I:\docs\maxim\prj\gis\learn\FloodBenineSmail\Maps\GDBs\FloodFactorsNormal.gdb\Transformed_HAND",
    # r"I:\docs\maxim\prj\gis\learn\FloodBenineSmail\Maps\GDBs\FloodFactorsNormal.gdb\hazard",
    # r"",
    # r"",
    
]
for raster in rasters:
    erase_from_raster(raster, clip)

--- Режим перезаписи: временно сохраняем в I:\docs\maxim\prj\gis\learn\FloodBenineSmail\Maps\GDBs\WorldCover.gdb\temp_manning_n_9dafae ---
--- Замена оригинального файла... ---
Готово: I:\docs\maxim\prj\gis\learn\FloodBenineSmail\Maps\GDBs\WorldCover.gdb\manning_n
--- Режим перезаписи: временно сохраняем в I:\docs\maxim\prj\gis\learn\FloodBenineSmail\Maps\GDBs\FloodFactors.gdb\temp_cost_horizont_5c6487 ---
--- Замена оригинального файла... ---
Готово: I:\docs\maxim\prj\gis\learn\FloodBenineSmail\Maps\GDBs\FloodFactors.gdb\cost_horizont
